# LR-QAOA vs WalkSAT benchmark (BM24)

Three training modes (set via `CFG["training_mode"]`):

- **`bm24_mean_p_fixed_n`** *(default)*: maximise mean p_succ at `train_n`.
  BM24-faithful baseline. Objective: `−mean_σ(p_succ(σ; β, γ))`.

- **`median_runtime_fixed_n`**: minimise median(1/(p_succ+ε)) at `train_n`.
  Robust runtime objective. Objective: `median_σ(1/(p_succ(σ)+ε))`.

- **`mean_log_runtime_fixed_n`**: minimise mean(ln(1/(p_succ+ε))) at `train_n`.
  Smoother than median runtime; less sensitive to single-instance spikes.

**Training rule**: one fixed `train_n` only. The n-slope on the eval plot is a
*diagnostic*, not the training objective. Evaluation runs on held-out instances
over `n_min … n_max` after training is complete.

**Eval**: log₂ slope of **median(1/p_succ)** vs n (lower = milder growth;
beat WalkSAT when LR slope < WalkSAT line).

**Plot**: depths where eval could not accept new angles are marked with **×**.

**Checkpoints**: each run writes `{run_stem}.partial.json` and `{run_stem}.json` under
`bm24_runs/MM-DD/runN/` after setup and after every depth (atomic writes). Set
`CFG["auto_resume"] = True` (default) and keep the same `run_stem` to continue after
a disconnect; or set `CFG["recover_from"]` to a specific checkpoint path.


In [ ]:
from pathlib import Path
import sys


def _notebook_bootstrap() -> None:
    """Put repo root + phasecraft + local experiment modules on sys.path."""
    here = Path.cwd().resolve()
    repo = here
    phasecraft = here / "phasecraft"
    for base in (here, *here.parents):
        pc = base / "phasecraft"
        if (pc / "lib" / "paths.py").is_file():
            repo, phasecraft = base, pc
            break
        if (base / "lib" / "paths.py").is_file() and base.name == "phasecraft":
            repo, phasecraft = base.parent, base
            break
    lr_scaling = phasecraft / "experiments" / "lr_scaling"
    sim = phasecraft / "lib" / "sim"
    for p in (repo, phasecraft, lr_scaling, sim):
        s = str(p)
        if s not in sys.path:
            sys.path.insert(0, s)


_notebook_bootstrap()

CFG = {
    # -- Problem (BM24 Random k-SAT) -- #
    "k": 8,
    "r": 176.54,
    "seed": 27,
    "depths": [2, 5, 8, 10, 15, 20, 30, 50, 70, 100],
    "lr_beta_schedule": "decreasing",

    # -- Training (single fixed n only) -- #
    # "bm24_mean_p_fixed_n"       : maximize mean(p_succ)              at train_n  [default / BM24 baseline]
    # "median_runtime_fixed_n"    : minimize median(1/(p_succ+eps))  at train_n  [robust runtime]
    # "mean_log_runtime_fixed_n"  : minimize mean(ln(1/(p_succ+eps))) at train_n  [smoother runtime]
    "training_mode": "bm24_mean_p_fixed_n",
    "train_n": 14,
    "train_size": 100,

    # -- Evaluation range -- #
    "n_min": 12,
    "n_max": 20,
    "test_size": 200,

    # -- COBYLA parameters -- #
    "skip_grid": False,
    "skip_grid_if_warm_start": False,
    "cobyla_maxiter": 160,
    "cobyla_restarts": 8,
    "cobyla_perturb_scale": 0.2,
    "grid_top_k": 5,

    # -- Classical baselines -- #
    # BM24 plain WalkSAT = pure random flip (p_noise=1.0), no greedy/break-count.
    # WalkSATlm: Cai-Luo-Su variant, p=0.15, w1=6, w2=5 (matches BM24 exactly).
    "walksat_p_noise": 1.0,
    "walksatlm_p_noise": 0.15,
    "walksatlm_w1": 6,
    "walksatlm_w2": 5,
    "max_flips": 100_000,

    # -- Eval / plot -- #
    "eval_axis": "runtime",
    "eval_aggregation": "median",
    "annotate_first_win": False,
    "eval_train_retries": 3,

    # -- Outputs / checkpoints -- #
    "output_dir": None,   # set below to canonical bm24_runs_dir()
    "run_stem": "scaling-tn14",
    "auto_resume": False,  # resume from latest checkpoint for run_stem
    "recover_from": None,  # or Path("bm24_runs/.../stem.partial.json")
    "use_dataset_cache": True,  # sidecar .json.gz beside checkpoint (clauses only)
}

from phasecraft.lib.sim.bm24_run_io import normalize_eval_aggregation, normalize_eval_axis
from phasecraft.lib.paths import bm24_runs_dir
from notebook_run_checkpoint import load_or_start_run

CFG["eval_axis"] = normalize_eval_axis(CFG.get("eval_axis", "runtime"))
CFG["eval_aggregation"] = normalize_eval_aggregation(CFG.get("eval_aggregation", "median"))
if CFG.get("output_dir") is None:
    CFG["output_dir"] = bm24_runs_dir()
else:
    CFG["output_dir"] = Path(CFG["output_dir"])
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
if CFG.get("recover_from") is not None:
    CFG["recover_from"] = Path(CFG["recover_from"])

run_stem, run_paths, trace, saved_setup = load_or_start_run(CFG)
print(f"Run id: {run_stem}")
print(f"Checkpoint dir: {run_paths['run_dir']}")
print(CFG)

In [2]:
import json
import time
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
from numba import njit
from scipy.stats import linregress
from tqdm.auto import tqdm

_notebook_bootstrap()

from phasecraft.lib.sim.bm24_qaoa_sim import (
    build_h_diagonal,
    generate_random_clause,
    generate_random_formula,
    make_lr_angles,
    per_instance_success_probability,
    run_qaoa,
)
from train_lr_fixed_n import (
    generate_training_instances,
    train_angles_fixed_n,
    SUPPORTED_TRAINING_MODES,
    DEFAULT_EVAL_RUNTIME_REGRESSION_FACTOR,
    DEFAULT_EVAL_TRAIN_RETRIES,
    eval_median_runtime_reject,
    run_train_eval_with_retries,
)

assert CFG["training_mode"] in SUPPORTED_TRAINING_MODES, (
    f"Unknown training_mode {CFG['training_mode']!r}. "
    f"Must be one of: {sorted(SUPPORTED_TRAINING_MODES)}"
)

K = int(CFG["k"])
LN2 = float(np.log(2.0))

In [ ]:
def generate_benchmark_dataset_cached(
    n_values,
    k: int,
    r: float,
    test_size: int,
    base_seed: int,
) -> Dict[int, List[dict]]:
    """SAT-filtered instances with cached H_diag (identical seeding to generate_benchmark_dataset)."""
    dataset: Dict[int, List[dict]] = {}
    for n in n_values:
        n = int(n)
        accepted = 0
        trial = 0
        instances = []
        pbar = tqdm(total=test_size, desc=f"dataset n={n}", leave=False)
        while accepted < test_size:
            if trial > 200_000:
                raise RuntimeError(f"too many rejections at n={n}")
            ss = np.random.SeedSequence([int(base_seed), n, accepted, trial])
            rng = np.random.default_rng(ss)
            lam = float(r) * n
            m_clauses = max(1, int(rng.poisson(lam)))
            clauses = [generate_random_clause(n, k, rng) for _ in range(m_clauses)]
            h_diag = build_h_diagonal(clauses, n)
            if not np.any(h_diag == 0):
                trial += 1
                continue
            instances.append({"clauses": clauses, "h_diag": h_diag})
            accepted += 1
            trial += 1
            pbar.update(1)
        pbar.close()
        dataset[n] = instances
    return dataset


def fit_log2_slope(n_values, y_values) -> float:
    n_arr = np.asarray(n_values, dtype=float)
    y_arr = np.asarray(y_values, dtype=float)
    mask = np.isfinite(y_arr) & (y_arr > 0)
    if mask.sum() < 2:
        return float("nan")
    slope_nat = linregress(n_arr[mask], np.log(y_arr[mask])).slope
    return float(slope_nat / LN2)


def evaluate_lr_qaoa_depth(dataset, n_values, betas, gammas, eps: float = 1e-300) -> dict:
    """Median(1/p_succ) per n using cached H_diag (BM24 run_qaoa)."""
    med_rt = {}
    for n in n_values:
        costs = []
        for inst in dataset[int(n)]:
            psi = run_qaoa(inst["h_diag"], betas, gammas, int(n))
            p = per_instance_success_probability(psi, inst["h_diag"])
            costs.append(1.0 / max(float(p), eps))
        med_rt[int(n)] = float(np.median(costs))
    return med_rt

In [ ]:
# --- One-time setup: benchmark data, classical baselines, training instances ---
from notebook_run_checkpoint import save_setup_checkpoint
from benchmark_dataset_cache import load_or_build_benchmark_dataset
from phasecraft.lib.sim.bm24_qaoa_sim import evaluate_walksat_on_dataset

t0 = time.time()
n_values = list(range(int(CFG["n_min"]), int(CFG["n_max"]) + 1))

dataset = load_or_build_benchmark_dataset(
    run_dir=run_paths["run_dir"],
    n_values=n_values,
    k=K,
    r=float(CFG["r"]),
    test_size=int(CFG["test_size"]),
    base_seed=int(CFG["seed"]),
    use_cache=bool(CFG.get("use_dataset_cache", True)),
    build_fn=lambda: generate_benchmark_dataset_cached(
        n_values, k=K, r=float(CFG["r"]),
        test_size=int(CFG["test_size"]), base_seed=int(CFG["seed"]),
    ),
)

if saved_setup and "classical" in saved_setup:
    classical = {
        algo: {int(n): float(v) for n, v in per_n.items()}
        for algo, per_n in saved_setup["classical"].items()
    }
    ws_slope = float(saved_setup["ws_slope"])
    lm_slope = float(saved_setup["lm_slope"])
    print("Restored classical baselines from checkpoint (skipping WalkSAT re-run)")
else:
    print("Classical solvers (once) — using standard bm24_qaoa_sim WalkSAT...")
    ws_res = evaluate_walksat_on_dataset(
        dataset, k=K, max_flips=int(CFG["max_flips"]),
        p_noise=float(CFG["walksat_p_noise"]), seed=int(CFG["seed"]), solver="walksat",
    )
    lm_res = evaluate_walksat_on_dataset(
        dataset, k=K, max_flips=int(CFG["max_flips"]),
        p_noise=float(CFG["walksatlm_p_noise"]), seed=int(CFG["seed"]), solver="walksatlm",
        walksatlm_w1=int(CFG["walksatlm_w1"]), walksatlm_w2=int(CFG["walksatlm_w2"]),
    )
    classical = {
        "walksat":   {int(n): v["median_flips"] for n, v in ws_res["per_n"].items()},
        "walksatlm": {int(n): v["median_flips"] for n, v in lm_res["per_n"].items()},
    }
    ws_slope = float(ws_res["fitted_exponents_log2"]["median_runtime_or_flips_log_slope"])
    lm_slope = float(lm_res["fitted_exponents_log2"]["median_runtime_or_flips_log_slope"])
print(f"  WalkSAT   (p_noise={CFG['walksat_p_noise']})   log2 slope = {ws_slope:.4f}")
print(f"  WalkSATlm (p_noise={CFG['walksatlm_p_noise']}) log2 slope = {lm_slope:.4f}")

print(
    f"Training instances (mode={CFG['training_mode']!r}, "
    f"n={CFG['train_n']}, size={CFG['train_size']})..."
)
training_instances = generate_training_instances(
    train_n=int(CFG["train_n"]),
    k=K,
    r=float(CFG["r"]),
    train_size=int(CFG["train_size"]),
    base_seed=int(CFG["seed"]),
)
print(f"  {len(training_instances)} SAT-filtered instances at n={CFG['train_n']}")
saved_setup = {
    "classical": {
        algo: {str(n): float(v) for n, v in per_n.items()}
        for algo, per_n in classical.items()
    },
    "ws_slope": float(ws_slope),
    "lm_slope": float(lm_slope),
}
setup_json = save_setup_checkpoint(
    run_paths,
    run_stem=run_stem,
    cfg=CFG,
    trace=trace,
    classical=classical,
    ws_slope=ws_slope,
    lm_slope=lm_slope,
)
print(f"Setup checkpoint -> {setup_json.name}")
print(f"Setup done in {time.time() - t0:.1f}s")

In [ ]:
# --- Depth sweep: train at fixed train_n + evaluate scaling across n ---
from notebook_run_checkpoint import build_payload, write_checkpoint


def _write_trace_checkpoint(trace_rows):
    payload = build_payload(
        run_stem=run_stem, cfg=CFG, trace=trace_rows, setup=saved_setup or None
    )
    return write_checkpoint(run_paths, payload)


depths_to_run = [int(p) for p in CFG["depths"]]
completed = {int(r["depth"]) for r in trace}
remaining = [d for d in depths_to_run if d not in completed]
print(f"Will run {len(depths_to_run)} depths: {depths_to_run[0]} … {depths_to_run[-1]}")
if completed:
    print(f"Already done: {sorted(completed)}")
    print(f"Remaining: {remaining}")

prev_deltas = None
prev_med_rt = None
prev_accepted_deltas = None
if trace:
    last = trace[-1]
    prev_deltas = (float(last["delta_gamma"]), float(last["delta_beta"]))
    prev_accepted_deltas = prev_deltas
    med = last.get("median_runtime_per_n")
    if med:
        prev_med_rt = {int(k): float(v) for k, v in med.items()}

_eval_rt_factor = float(
    CFG.get("eval_runtime_regression_factor", DEFAULT_EVAL_RUNTIME_REGRESSION_FACTOR)
)

for depth in remaining:
    depth = int(depth)
    t_depth = time.time()
    print(f"\n{'=' * 60}\nDepth p = {depth}\n{'=' * 60}")

    def _train_at_depth(retry_index: int):
        # On retries: use warm start (prev accepted), always run grid.
        warm = (
            tuple(prev_accepted_deltas)
            if retry_index > 0 and prev_accepted_deltas is not None
            else (tuple(prev_deltas) if prev_deltas is not None else None)
        )
        perturb = float(CFG["cobyla_perturb_scale"]) * (1.25 ** int(retry_index))
        train_rng = np.random.default_rng(
            int(CFG["seed"]) + 1000 + depth + 10_000 * int(retry_index)
        )
        _, diag_local = train_angles_fixed_n(
            training_mode=str(CFG["training_mode"]),
            train_n=int(CFG["train_n"]),
            depth=depth,
            instances=training_instances,
            initial_angles=warm,
            beta_schedule=str(CFG["lr_beta_schedule"]),
            # On retries always run the grid (skip flags only apply on first attempt).
            skip_grid=bool(CFG["skip_grid"]) and retry_index == 0,
            skip_grid_if_warm_start=bool(CFG.get("skip_grid_if_warm_start", True)) and retry_index == 0,
            cobyla_maxiter=int(CFG["cobyla_maxiter"]),
            cobyla_restarts=int(CFG["cobyla_restarts"]),
            cobyla_perturb_scale=perturb,
            grid_top_k=int(CFG["grid_top_k"]),
            rng=train_rng,
        )
        dg_l = float(diag_local["best_deltas"][0])
        db_l = float(diag_local["best_deltas"][1])
        print(
            f"  trained dg={dg_l:.6f} db={db_l:.6f} "
            f"mean_p@train_n={diag_local['best_avg_train_p_succ']:.4e}"
        )
        if diag_local.get("anti_regression_applied"):
            print("  (anti-regression: kept warm-start)")
        if diag_local.get("train_rejected"):
            print(
                f"  (collapse guard: {diag_local.get('train_reject_reason', '')}; "
                f"applied={diag_local.get('collapse_guard_applied', False)})"
            )
        return dg_l, db_l, diag_local

    def _evaluate_at_angles(dg_eval: float, db_eval: float):
        # Evaluation runs AFTER training, over all n_values. This is where the
        # scaling slope is measured — it is never fed back into the training objective.
        betas_e, gammas_e = make_lr_angles(
            dg_eval, db_eval, depth,
            beta_schedule=str(CFG["lr_beta_schedule"]),
            angle_convention="bm24",
        )
        return evaluate_lr_qaoa_depth(dataset, n_values, betas_e, gammas_e)

    te_result = run_train_eval_with_retries(
        train_at_depth=_train_at_depth,
        evaluate_at_angles=_evaluate_at_angles,
        prev_med_rt=prev_med_rt,
        prev_accepted_deltas=prev_accepted_deltas,
        n_min=int(CFG["n_min"]),
        n_max=int(CFG["n_max"]),
        max_retries=int(CFG.get("eval_train_retries", DEFAULT_EVAL_TRAIN_RETRIES)),
        regression_factor=_eval_rt_factor,
    )
    dg, db = te_result["dg"], te_result["db"]
    diag = te_result["diag"]
    med_rt = te_result["med_rt"]
    eval_rejected = bool(te_result["eval_rejected"])
    reject_reason = te_result["eval_reject_reason"]

    trained_dg, trained_db = (
        float(diag.get("trained_deltas", diag["best_deltas"])[0]),
        float(diag.get("trained_deltas", diag["best_deltas"])[1]),
    )
    if not eval_rejected:
        prev_accepted_deltas = (dg, db)
        prev_med_rt = dict(med_rt)

    prev_deltas = prev_accepted_deltas
    lr_slope = fit_log2_slope(n_values, [med_rt[n] for n in n_values])
    beats_ws = bool(np.isfinite(lr_slope) and np.isfinite(ws_slope) and lr_slope < ws_slope)
    beats_lm = bool(np.isfinite(lr_slope) and np.isfinite(lm_slope) and lr_slope < lm_slope)

    row = {
        "depth": depth,
        "delta_gamma": float(dg),
        "delta_beta": float(db),
        "trained_delta_gamma": float(trained_dg),
        "trained_delta_beta": float(trained_db),
        "training_mode": str(CFG["training_mode"]),
        "training_objective": diag.get("objective"),
        "train_rejected": bool(diag.get("train_rejected", False)),
        "train_reject_reason": diag.get("train_reject_reason"),
        "collapse_guard_applied": bool(diag.get("collapse_guard_applied", False)),
        "eval_rejected": bool(eval_rejected),
        "eval_reject_reason": reject_reason if eval_rejected else None,
        "eval_train_attempts": int(te_result.get("eval_train_attempts", 1)),
        "used_previous_angles": bool(te_result.get("used_previous_angles", False)),
        "training_failed": bool(
            te_result.get("used_previous_angles") or te_result.get("eval_rejected")
        ),
        "best_train_objective": diag.get("best_train_objective"),
        "best_avg_train_p_succ": diag.get("best_avg_train_p_succ"),
        "lr_log2_slope": lr_slope,
        "walksat_log2_slope": ws_slope,
        "walksatlm_log2_slope": lm_slope,
        "median_runtime_per_n": {str(n): med_rt[n] for n in n_values},
        "lr_beats_walksat_scaling": beats_ws,
        "lr_beats_walksatlm_scaling": beats_lm,
        "elapsed_s": time.time() - t_depth,
    }
    trace.append(row)
    print(f"  LR log2 slope={lr_slope:.4f}  (WS={ws_slope:.4f} LM={lm_slope:.4f})")
    print(f"  beats WS={beats_ws} beats LM={beats_lm}  elapsed={row['elapsed_s']:.1f}s")
    out_json = _write_trace_checkpoint(trace)
    print(f"  checkpoint -> {out_json.name}")

if not remaining:
    out_json = _write_trace_checkpoint(trace)
    print("Nothing left to run — trace already complete.")

print(f"\nWrote {out_json}")


Will run 11 depths: 2 … 100

Depth p = 2
  > train_angles_fixed_n: mode='bm24_mean_p_fixed_n' n=14 depth=2 instances=100


Grid (dg): 100%|██████████| 11/11 [00:31<00:00,  2.82s/it]


  > Best grid: dg=-1.8010 db=+1.2700 obj=-0.002352
  > COBYLA restart  0: obj=-0.002360 x=[-1.7557, 1.2974]
  > COBYLA restart  1: obj=-0.002360 x=[-1.7564, 1.2974]
  > COBYLA restart  5: obj=-0.002360 x=[-1.7562, 1.2972]
  > COBYLA restart  6: obj=-0.002360 x=[-1.7569, 1.2974]
  > Result: dg=-1.756205 db=+1.297237 obj=-0.002360 mean_p@train_n=2.3601e-03
  trained dg=-1.756205 db=1.297237 mean_p@train_n=2.3601e-03
  LR log2 slope=0.6767  (WS=0.3370 LM=0.2911)
  beats WS=False beats LM=False  elapsed=207.3s
  checkpoint -> scaling-tn14.json

Depth p = 5
  > train_angles_fixed_n: mode='bm24_mean_p_fixed_n' n=14 depth=5 instances=100
  > Warm-start: dg=-1.7562 db=+1.2972 obj=-0.011330


Grid (dg): 100%|██████████| 11/11 [01:11<00:00,  6.53s/it]


  > Best grid: dg=-1.6020 db=+1.2700 obj=-0.012067
  > COBYLA restart  0: obj=-0.012989 x=[-1.6587, 1.0525]
  > COBYLA restart  7: obj=-0.012989 x=[-1.6587, 1.0528]
  > Result: dg=-1.658676 db=+1.052767 obj=-0.012989 mean_p@train_n=1.2989e-02
  trained dg=-1.658676 db=1.052767 mean_p@train_n=1.2989e-02
  LR log2 slope=0.5679  (WS=0.3370 LM=0.2911)
  beats WS=False beats LM=False  elapsed=524.8s
  checkpoint -> scaling-tn14.json

Depth p = 8
  > train_angles_fixed_n: mode='bm24_mean_p_fixed_n' n=14 depth=8 instances=100
  > Warm-start: dg=-1.6587 db=+1.0528 obj=-0.029633


...

  > Best grid: dg=-1.6020 db=+0.8800 obj=-0.073734
  > COBYLA restart  0: obj=-0.077720 x=[-1.5378, 1.0168]
  > COBYLA restart  1: obj=-0.077720 x=[-1.5367, 1.0167]
  > COBYLA restart  2: obj=-0.077720 x=[-1.5373, 1.0173]
  > COBYLA restart  4: obj=-0.077720 x=[-1.5367, 1.0174]
  > COBYLA restart  5: obj=-0.077720 x=[-1.5368, 1.0173]
  > COBYLA restart  7: obj=-0.077720 x=[-1.5366, 1.0171]
  > Result: dg=-1.536647 db=+1.017084 obj=-0.077720 mean_p@train_n=7.7720e-02
  trained dg=-1.536647 db=1.017084 mean_p@train_n=7.7720e-02
  LR log2 slope=0.4771  (WS=0.3370 LM=0.2911)
  beats WS=False beats LM=False  elapsed=199077.8s
  checkpoint -> scaling-tn14.json

Depth p = 20
  > train_angles_fixed_n: mode='bm24_mean_p_fixed_n' n=14 depth=20 instances=100
  > Warm-start: dg=-1.5366 db=+1.0171 obj=-0.107438


Grid (dg): 100%|██████████| 11/11 [10:38:53<00:00, 3484.89s/it] 


  > Best grid: dg=-1.6020 db=+0.8800 obj=-0.106918
  > COBYLA restart  0: obj=-0.109326 x=[-1.5194, 0.9404]
  > COBYLA restart  1: obj=-0.109327 x=[-1.5185, 0.9406]
  > COBYLA restart  4: obj=-0.109327 x=[-1.5185, 0.9409]
  > Result: dg=-1.518507 db=+0.940915 obj=-0.109327 mean_p@train_n=1.0933e-01
  trained dg=-1.518507 db=0.940915 mean_p@train_n=1.0933e-01
  LR log2 slope=0.4398  (WS=0.3370 LM=0.2911)
  beats WS=False beats LM=False  elapsed=88459.3s
  checkpoint -> scaling-tn14.json

Depth p = 30
  > train_angles_fixed_n: mode='bm24_mean_p_fixed_n' n=14 depth=30 instances=100
  > Warm-start: dg=-1.5185 db=+0.9409 obj=-0.159380


Grid (dg): 100%|██████████| 11/11 [4:36:44<00:00, 1509.48s/it] 


  > Best grid: dg=-1.4030 db=+0.8800 obj=-0.158036
  > COBYLA restart  0: obj=-0.161625 x=[-1.3507, 0.9975]
  > COBYLA restart  1: obj=-0.161626 x=[-1.3531, 0.9969]
  > COBYLA restart  3: obj=-0.161626 x=[-1.3524, 0.9972]
  > Result: dg=-1.352363 db=+0.997216 obj=-0.161626 mean_p@train_n=1.6163e-01
  trained dg=-1.352363 db=0.997216 mean_p@train_n=1.6163e-01


In [ ]:
# --- Optional: plot from saved JSON only (no re-run) ---
# Set plot_json to a finished .json if you only want the graph.
plot_json = None  # e.g. Path("bm24_runs/05-29_1922-efficient-scaling.json")

if plot_json is not None:
    with open(plot_json, encoding="utf-8") as f:
        saved = json.load(f)
    trace = saved["trace"]
    run_stem = saved.get("run_stem", Path(plot_json).stem)
    if trace and "walksat_log2_slope" in trace[0]:
        ws_slope = trace[0]["walksat_log2_slope"]
        lm_slope = trace[0]["walksatlm_log2_slope"]
    else:
        raise RuntimeError("JSON trace missing classical slopes; run setup cell first")
    print(f"Loaded {len(trace)} depths from {plot_json}")
else:
    print("plot_json is None — run the summary plot cell below after the sweep finishes")

In [ ]:
# --- Summary plot (same y as pipeline scaling-vs-depth) ---
from bm24_run_io import plot_scaling_vs_depth, scaling_plot_headline
from IPython.display import Image, display

win_depths = [
    row["depth"]
    for row in trace
    if row.get("lr_beats_walksat_scaling") and row.get("lr_beats_walksatlm_scaling")
]
if win_depths and not CFG.get("annotate_first_win", False):
    print(
        f"(info) first depth beating both classical slopes: p={min(win_depths)} "
        "(annotate_first_win=False)"
    )
plot_path = plot_scaling_vs_depth(
    trace,
    run_paths["png"],
    settings={**CFG, "k": K},
    headline=scaling_plot_headline(
        "Efficient LR sweep", CFG["eval_axis"], CFG["eval_aggregation"]
    ),
    annotate_first_win=bool(CFG.get("annotate_first_win", False)),
)
display(Image(filename=str(plot_path)))
print(f"Saved {plot_path}")